# Every Character Counts — Colab launcher

This notebook only installs the package and calls the registered CLI. It does not duplicate research logic. Run the cells in order and stop after the smoke-test gate unless the gate passes.

In [1]:
# 1. Mount persistent storage and read the private HF token.
from google.colab import drive, userdata
from pathlib import Path
import os

drive.mount('/content/drive')
token = userdata.get('HF_TOKEN')
if not token:
    raise RuntimeError('Add an HF_TOKEN secret and enable notebook access.')
os.environ['HF_TOKEN'] = token
os.environ['HUGGING_FACE_HUB_TOKEN'] = token

REPO_URL = 'https://github.com/CherryWang77/classical-chinese-poetry-lm.git'
BRANCH = 'research/every-character-counts'
REPO = Path('/content/every-character-counts')
DRIVE_ROOT = Path('/content/drive/MyDrive/every-character-counts')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(DRIVE_ROOT / 'hf_cache')
print('Persistent root:', DRIVE_ROOT)

Mounted at /content/drive
Persistent root: /content/drive/MyDrive/every-character-counts


In [9]:
# 2. Obtain a clean ephemeral checkout for this runtime.
import shutil, subprocess

if REPO.exists():
    shutil.rmtree(REPO)  # /content is ephemeral; persistent results are on Drive.
subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(REPO)], check=True)
print(subprocess.run(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], check=True, capture_output=True, text=True).stdout.strip())

ea673eac7dbd921f2f85946badf62343c41121d2


In [10]:
# 3. Link only research data/artifacts/results to Google Drive.
def persistent_link(relative: str) -> None:
    local = REPO / relative
    target = DRIVE_ROOT / relative
    target.mkdir(parents=True, exist_ok=True)
    local.parent.mkdir(parents=True, exist_ok=True)
    if local.is_symlink() or local.is_file():
        local.unlink()
    elif local.exists():
        shutil.rmtree(local)
    local.symlink_to(target, target_is_directory=True)

for path in ('data/research', 'artifacts/research', 'results/research', 'output/pdf'):
    persistent_link(path)
print('Drive persistence links are ready.')

Drive persistence links are ready.


In [11]:
# 4. Install and define the CLI helper.
subprocess.run(['python', '-m', 'pip', 'install', '-q', '-e', f'{REPO}[research,dev,report]'], check=True)

def poetry(*args: str) -> None:
    command = ['poetry-lm', '--project-root', str(REPO), *map(str, args)]
    print('>', ' '.join(command))
    subprocess.run(command, cwd=REPO, check=True, env=os.environ.copy())

In [12]:
# 5. Mandatory T4 and environment preflight.
import torch
subprocess.run(['nvidia-smi'], check=True)
assert torch.cuda.is_available(), 'No CUDA GPU: Runtime > Change runtime type > T4 GPU'
props = torch.cuda.get_device_properties(0)
print('GPU:', torch.cuda.get_device_name(0))
print('CUDA:', torch.version.cuda)
print('VRAM GiB:', round(props.total_memory / 2**30, 2))
assert props.total_memory >= 14 * 2**30, 'At least a 16 GB-class GPU is required.'

GPU: Tesla T4
CUDA: 12.8
VRAM GiB: 14.56


In [13]:
# 6. Rebuild the pinned dataset and run all offline tests.
poetry('data', 'build', '--config', str(REPO / 'configs/research/data.json'), '--fetch')
subprocess.run(['python', '-m', 'pytest', '-q'], cwd=REPO, check=True)
subprocess.run(['ruff', 'check', 'src', 'tests'], cwd=REPO, check=True)

> poetry-lm --project-root /content/every-character-counts data build --config /content/every-character-counts/configs/research/data.json --fetch


CompletedProcess(args=['ruff', 'check', 'src', 'tests'], returncode=0)

## Mandatory GPU smoke test
The next three cells train two optimizer steps on 20 examples, generate 20 constrained poems, and enforce 100% structural validity with zero dead ends. Do not run formal training if any assertion fails.

In [7]:
# 7. Tiny QLoRA smoke training; safe to rerun after a disconnect.
poetry('train', 'lora', '--config', str(REPO / 'configs/research/smoke/lora_t4.json'), '--seed', '42', '--resume', 'auto')

> poetry-lm --project-root /content/every-character-counts train lora --config /content/every-character-counts/configs/research/smoke/lora_t4.json --seed 42 --resume auto


In [14]:
# 8. Generate the registered 10 templates × 2 themes smoke grid.
poetry('generate', 'benchmark', '--config', str(REPO / 'configs/research/smoke/benchmark_t4.json'), '--system', 'qwen_lora_hard_structure', '--training-seed', '42', '--split', 'validation')
poetry('evaluate', '--manifest', str(REPO / 'results/research/smoke/generations.jsonl'), '--config', str(REPO / 'configs/research/smoke/benchmark_t4.json'), '--output-dir', str(REPO / 'results/research/smoke/evaluation'))

> poetry-lm --project-root /content/every-character-counts generate benchmark --config /content/every-character-counts/configs/research/smoke/benchmark_t4.json --system qwen_lora_hard_structure --training-seed 42 --split validation
> poetry-lm --project-root /content/every-character-counts evaluate --manifest /content/every-character-counts/results/research/smoke/generations.jsonl --config /content/every-character-counts/configs/research/smoke/benchmark_t4.json --output-dir /content/every-character-counts/results/research/smoke/evaluation


In [15]:
# 9. Release gate. The final line must print SMOKE TEST PASSED.
import json
summary = json.loads((REPO / 'results/research/smoke/evaluation/summary.json').read_text())
metrics = summary['splits']['validation']['systems']['qwen_lora_hard_structure']
assert metrics['outputs'] == 20, metrics
assert metrics['structural_exact_match'] == 1.0, metrics
assert metrics['constraint_dead_end_rate'] == 0.0, metrics
print('SMOKE TEST PASSED')

SMOKE TEST PASSED


## Formal runs — only after approval
Run one seed per Colab session. Set `FORMAL_SEED` to 42, 1729, or 2026. Rerunning the same cell uses the latest persistent checkpoint.

In [ ]:
FORMAL_SEED = 42

poetry(
    "train", "lora",
    "--config", str(REPO / "configs/research/lora_train.json"),
    "--seed", str(FORMAL_SEED),
    "--resume", "auto",
)

> poetry-lm --project-root /content/every-character-counts train lora --config /content/every-character-counts/configs/research/lora_train.json --seed 42 --resume auto


In [19]:
# 10. Formal QLoRA: run one seed per session.
FORMAL_SEED = 42  # Change only to 1729 or 2026 on later sessions.
assert FORMAL_SEED in (42, 1729, 2026)
poetry('train', 'lora', '--config', str(REPO / 'configs/research/lora_train.json'), '--seed', str(FORMAL_SEED), '--resume', 'auto')

> poetry-lm --project-root /content/every-character-counts train lora --config /content/every-character-counts/configs/research/lora_train.json --seed 42 --resume auto


KeyboardInterrupt: 

In [ ]:
# 11. Validation-only lambda sweep for the current trained seed.
validation_manifest = REPO / 'results/research/validation_rhyme_sweep.jsonl'
for value in (0.0, 0.5, 1.0, 2.0, 4.0):
    poetry('generate', 'benchmark', '--config', str(REPO / 'configs/research/benchmark.json'), '--system', 'qwen_lora_hard_structure_rhyme', '--training-seed', str(FORMAL_SEED), '--split', 'validation', '--rhyme-lambda', str(value), '--output', str(validation_manifest))

In [ ]:
# 12. Run only after all three seeds completed the five-value validation sweep.
poetry('evaluate', '--manifest', str(validation_manifest), '--select-rhyme', '--selection-output', str(REPO / 'results/research/selected_rhyme_lambda.json'))

In [ ]:
# 13. Formal test systems for the current seed. Never run before cell 12 succeeds.
test_manifest = REPO / 'results/research/generations.jsonl'
for system in ('qwen_lora_unconstrained', 'qwen_lora_best_of_8', 'qwen_lora_hard_structure', 'qwen_lora_hard_structure_rhyme'):
    poetry('generate', 'benchmark', '--config', str(REPO / 'configs/research/benchmark.json'), '--system', system, '--training-seed', str(FORMAL_SEED), '--split', 'test', '--output', str(test_manifest))

In [ ]:
# 14. Zero-shot baseline runs once, not once per training seed.
poetry('generate', 'benchmark', '--config', str(REPO / 'configs/research/benchmark.json'), '--system', 'qwen_zero_shot', '--split', 'test', '--output', str(test_manifest))

In [ ]:
# 15. Final automatic evaluation after every registered generation is complete.
poetry('evaluate', '--manifest', str(test_manifest), '--config', str(REPO / 'configs/research/benchmark.json'), '--output-dir', str(REPO / 'results/research/evaluation'))